# Product Image Classifier — Exploration & Experimentation

This notebook walks through the complete ML workflow:
1. **Data Exploration** — Understand your dataset
2. **Preprocessing Visualization** — See what augmentation does
3. **Model Architecture** — Inspect the network
4. **Training Experiment** — Train locally and visualize learning curves
5. **Evaluation Deep Dive** — Confusion matrix, error analysis
6. **SageMaker Integration** — Launch cloud training

---

## Why This Notebook Matters for Interviews

FAANG interviewers often ask: *"Walk me through your ML workflow."*
This notebook IS that walkthrough — from raw data to deployed model.
Keep it in the repo as evidence of your systematic approach.

In [ ]:
# ============================================================================
# Setup & Imports
# ============================================================================
import sys
sys.path.insert(0, '../src/training')
sys.path.insert(0, '../src/inference')

import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from collections import Counter
from pathlib import Path
import json
import os

# Our modules
from model import ProductClassifier, get_model
from dataset import (
    ProductImageDataset, 
    get_train_transforms, 
    get_val_transforms,
    create_data_loaders
)

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Data Exploration

Before training any model, we MUST understand our data.

**Key questions:**
- How many images per category? (class balance)
- What do the images look like? (quality, variety)
- Are there any obvious problems? (corrupted images, wrong labels)

In [ ]:
# ============================================================================
# Dataset Statistics
# ============================================================================
# CONCEPT: Always visualize your data distribution FIRST.
# Class imbalance is the #1 silent killer of ML model quality.

DATA_DIR = '../data/train'  # Update this path to your training data
CATEGORIES = ['electronics', 'clothing', 'furniture', 'books', 'toys']

# Count images per category
class_counts = {}
for cat in CATEGORIES:
    cat_dir = Path(DATA_DIR) / cat
    if cat_dir.exists():
        count = len([f for f in cat_dir.iterdir() if f.suffix.lower() in {'.jpg', '.jpeg', '.png'}])
        class_counts[cat] = count
    else:
        class_counts[cat] = 0
        print(f'WARNING: {cat_dir} does not exist!')

# Plot distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']
bars = ax1.bar(class_counts.keys(), class_counts.values(), color=colors)
ax1.set_title('Images per Category', fontweight='bold')
ax1.set_ylabel('Count')
for bar, count in zip(bars, class_counts.values()):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 10,
             str(count), ha='center', fontweight='bold')

# Pie chart
ax2.pie(class_counts.values(), labels=class_counts.keys(), colors=colors,
        autopct='%1.1f%%', startangle=90)
ax2.set_title('Class Distribution', fontweight='bold')

plt.tight_layout()
plt.show()

total = sum(class_counts.values())
print(f'\nTotal images: {total}')
if total > 0:
    max_count = max(class_counts.values())
    min_count = min(class_counts.values())
    print(f'Imbalance ratio: {max_count/max(min_count,1):.1f}x')
    if max_count > 3 * min_count:
        print('⚠️ SIGNIFICANT CLASS IMBALANCE — consider weighted loss or oversampling')

In [ ]:
# ============================================================================
# Sample Image Grid
# ============================================================================
# CONCEPT: Always look at your actual data. You might find:
# - Mislabeled images (a book in the electronics folder)
# - Low quality images (blurry, too dark)
# - Non-product images (lifestyle shots, text-only images)

def show_sample_images(data_dir, categories, samples_per_cat=4):
    """Display a grid of sample images from each category."""
    fig, axes = plt.subplots(len(categories), samples_per_cat, 
                             figsize=(3*samples_per_cat, 3*len(categories)))
    
    for i, cat in enumerate(categories):
        cat_dir = Path(data_dir) / cat
        if not cat_dir.exists():
            continue
        images = sorted(cat_dir.iterdir())[:samples_per_cat]
        
        for j in range(samples_per_cat):
            ax = axes[i][j] if len(categories) > 1 else axes[j]
            if j < len(images):
                try:
                    img = Image.open(images[j]).convert('RGB')
                    ax.imshow(img)
                    if j == 0:
                        ax.set_ylabel(cat, fontsize=14, fontweight='bold')
                except Exception as e:
                    ax.text(0.5, 0.5, f'Error:\n{e}', ha='center', va='center')
            ax.set_xticks([])
            ax.set_yticks([])
    
    plt.suptitle('Sample Images per Category', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

if total > 0:
    show_sample_images(DATA_DIR, CATEGORIES)

## 2. Data Augmentation Visualization

**CONCEPT:** Data augmentation creates "new" training samples by randomly
transforming existing images. Let's see what each transform actually does.

In [ ]:
# ============================================================================
# Visualize Augmentation Effects
# ============================================================================

def show_augmentations(image_path, num_augmented=8):
    """Show original image alongside augmented versions."""
    img = Image.open(image_path).convert('RGB')
    train_transform = get_train_transforms()
    
    fig, axes = plt.subplots(2, (num_augmented + 2) // 2, figsize=(16, 7))
    axes = axes.flatten()
    
    # Show original
    axes[0].imshow(img)
    axes[0].set_title('ORIGINAL', fontweight='bold', color='red')
    axes[0].set_xticks([]); axes[0].set_yticks([])
    
    # Show augmented versions
    for i in range(1, num_augmented + 1):
        augmented_tensor = train_transform(img)
        # Denormalize for display
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        display_tensor = augmented_tensor * std + mean
        display_tensor = display_tensor.clamp(0, 1)
        
        if i < len(axes):
            axes[i].imshow(display_tensor.permute(1, 2, 0).numpy())
            axes[i].set_title(f'Augmented #{i}')
            axes[i].set_xticks([]); axes[i].set_yticks([])
    
    # Hide unused axes
    for j in range(num_augmented + 1, len(axes)):
        axes[j].set_visible(False)
    
    plt.suptitle('Same Image → Different Augmentations Each Time', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Show augmentation on a sample image (update path as needed)
sample_images = list(Path(DATA_DIR).rglob('*.jpg'))[:1]
if sample_images:
    show_augmentations(sample_images[0])
else:
    print('No images found — create a dummy image for demo')
    dummy = Image.new('RGB', (300, 300), color='blue')
    dummy.save('/tmp/demo.jpg')
    show_augmentations('/tmp/demo.jpg')

## 3. Model Architecture Inspection

Let's look at what we're working with — layer structure, parameter counts,
and which layers are frozen vs. trainable.

In [ ]:
# ============================================================================
# Model Summary
# ============================================================================

model = ProductClassifier(num_classes=5, pretrained=True, freeze_layers=6)

print('=' * 60)
print('MODEL SUMMARY')
print('=' * 60)
print(f'Architecture: ResNet18 + Custom Classification Head')
print(f'Total parameters:     {model.get_total_params():>12,}')
print(f'Trainable parameters: {model.get_trainable_params():>12,}')
print(f'Frozen parameters:    {model.get_total_params() - model.get_trainable_params():>12,}')
print(f'Trainable ratio:      {model.get_trainable_params()/model.get_total_params()*100:.1f}%')
print(f'Input size:           {model.INPUT_SIZE}')
print(f'Categories:           {model.CATEGORIES}')

print('\n' + '=' * 60)
print('LAYER-BY-LAYER FREEZE STATUS')
print('=' * 60)
for name, param in model.named_parameters():
    status = '🔒 FROZEN' if not param.requires_grad else '🔓 TRAINABLE'
    print(f'  {status}  {name:<45} {param.numel():>10,} params')

In [ ]:
# ============================================================================
# Quick Forward Pass Test
# ============================================================================
# CONCEPT: Always test your model with a dummy input before training.
# This catches shape mismatches, device issues, and architecture bugs.

dummy_input = torch.randn(1, 3, 224, 224)
model.eval()

with torch.no_grad():
    output = model(dummy_input)
    probs = torch.nn.functional.softmax(output, dim=1)

print(f'Input shape:  {dummy_input.shape}')
print(f'Output shape: {output.shape}')
print(f'\nRaw logits:       {output[0].numpy()}')
print(f'Probabilities:    {probs[0].numpy()}')
print(f'Sum of probs:     {probs[0].sum().item():.6f} (should be ~1.0)')
print(f'Predicted class:  {model.CATEGORIES[probs[0].argmax()]} ({probs[0].max():.2%})')
print('\n✅ Model forward pass working correctly!')

## 4. Training Experiment (Local)

Train a quick experiment locally to validate the pipeline.
For production training, use SageMaker (Section 6).

In [ ]:
# ============================================================================
# Local Training Loop (abbreviated for notebook)
# ============================================================================
# NOTE: This uses the same logic as src/training/train.py
# but with visualization added.

# Skip if no training data available
if total == 0:
    print('No training data found. Skipping training demo.')
    print('Upload images to ../data/train/{category}/ to enable this cell.')
else:
    EPOCHS = 5
    BATCH_SIZE = 16
    LR = 0.001
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = ProductClassifier(num_classes=5, pretrained=True)
    model.to(device)
    
    train_loader, val_loader = create_data_loaders(
        train_dir='../data/train',
        val_dir='../data/val',
        batch_size=BATCH_SIZE,
        num_workers=2
    )
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=LR
    )
    
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    
    for epoch in range(EPOCHS):
        # Train
        model.train()
        running_loss, correct, total_samples = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, pred = outputs.max(1)
            total_samples += labels.size(0)
            correct += pred.eq(labels).sum().item()
        
        train_loss = running_loss / len(train_loader)
        train_acc = 100. * correct / total_samples
        
        # Validate
        model.eval()
        val_loss_sum, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                val_loss_sum += criterion(outputs, labels).item()
                _, pred = outputs.max(1)
                val_total += labels.size(0)
                val_correct += pred.eq(labels).sum().item()
        
        val_loss = val_loss_sum / len(val_loader)
        val_acc = 100. * val_correct / val_total
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        print(f'Epoch {epoch+1}/{EPOCHS} | '
              f'Train Loss: {train_loss:.4f}, Acc: {train_acc:.1f}% | '
              f'Val Loss: {val_loss:.4f}, Acc: {val_acc:.1f}%')

In [ ]:
# ============================================================================
# Plot Learning Curves
# ============================================================================
# CONCEPT: Learning curves show if training is working properly.
#
# Healthy training:
#   - Both train & val loss decrease
#   - Small gap between train & val accuracy
#
# Overfitting signs:
#   - Train loss keeps decreasing, val loss starts INCREASING
#   - Large gap: train acc 99%, val acc 70%
#
# Underfitting signs:
#   - Both losses stay high / plateau early
#   - Both accuracies are low

if 'history' in dir() and len(history['train_loss']) > 0:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    epochs_range = range(1, len(history['train_loss']) + 1)
    
    # Loss
    ax1.plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss')
    ax1.plot(epochs_range, history['val_loss'], 'r-o', label='Val Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training & Validation Loss', fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Accuracy
    ax2.plot(epochs_range, history['train_acc'], 'b-o', label='Train Accuracy')
    ax2.plot(epochs_range, history['val_acc'], 'r-o', label='Val Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title('Training & Validation Accuracy', fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim([0, 105])
    
    plt.tight_layout()
    plt.show()
    
    # Overfitting check
    gap = history['train_acc'][-1] - history['val_acc'][-1]
    if gap > 15:
        print(f'⚠️ Overfitting detected! Train-Val gap: {gap:.1f}%')
        print('   Try: more dropout, more augmentation, or early stopping')
    elif gap < 2:
        print(f'✅ Good generalization. Train-Val gap: {gap:.1f}%')
    else:
        print(f'ℹ️ Moderate gap: {gap:.1f}%. Training looks reasonable.')
else:
    print('No training history to plot. Run the training cell above first.')

## 5. Evaluation Deep Dive

Accuracy alone is not enough — we need per-class metrics and a confusion matrix
to understand WHERE the model fails.

In [ ]:
# ============================================================================
# Confusion Matrix Visualization
# ============================================================================
# CONCEPT: A confusion matrix shows prediction patterns.
# Diagonal = correct predictions (we want these to be high)
# Off-diagonal = errors (shows which classes get confused)

def plot_confusion_matrix(cm, categories):
    """Plot a beautiful confusion matrix."""
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Normalize by row (true labels)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    
    # Labels
    ax.set_xticks(range(len(categories)))
    ax.set_yticks(range(len(categories)))
    ax.set_xticklabels(categories, rotation=45, ha='right')
    ax.set_yticklabels(categories)
    ax.set_xlabel('Predicted', fontsize=14, fontweight='bold')
    ax.set_ylabel('Actual', fontsize=14, fontweight='bold')
    ax.set_title('Confusion Matrix (Normalized)', fontsize=16, fontweight='bold')
    
    # Annotate cells
    for i in range(len(categories)):
        for j in range(len(categories)):
            color = 'white' if cm_norm[i][j] > 0.5 else 'black'
            ax.text(j, i, f'{cm[i][j]}\n({cm_norm[i][j]:.0%})',
                    ha='center', va='center', color=color, fontsize=11)
    
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

# Example with dummy data (replace with real evaluation results)
example_cm = np.array([
    [95,  2,  1,  1,  1],
    [ 1, 88,  3,  5,  3],
    [ 2,  4, 90,  2,  2],
    [ 3,  5,  2, 87,  3],
    [ 1,  3,  1,  2, 93]
])

plot_confusion_matrix(example_cm, CATEGORIES)
print('Key observations:')
print('  - Electronics and Toys are easiest to classify')
print('  - Clothing ↔ Books confusion: both rectangular with text?')
print('  - This insight could guide data collection for improvement')

## 6. SageMaker Integration

Launch training on SageMaker for production-grade model training.

In [ ]:
# ============================================================================
# SageMaker Training Job (uncomment to run)
# ============================================================================
# CONCEPT: SageMaker training provisions a GPU instance, trains your model,
# saves artifacts to S3, and terminates the instance. You only pay for
# the time the instance is running.

'''
import sagemaker
from sagemaker.pytorch import PyTorch

session = sagemaker.Session()
role = sagemaker.get_execution_role()

estimator = PyTorch(
    entry_point='train.py',
    source_dir='../src/training/',
    role=role,
    framework_version='2.0.0',
    py_version='py310',
    instance_count=1,
    instance_type='ml.g4dn.xlarge',
    use_spot_instances=True,
    max_wait=7200,
    max_run=3600,
    hyperparameters={
        'epochs': 15,
        'batch-size': 32,
        'learning-rate': 0.001,
        'weight-decay': 0.0001,
        'dropout-rate': 0.5,
        'patience': 5,
    }
)

estimator.fit({
    'train': 's3://your-bucket/train/',
    'validation': 's3://your-bucket/val/',
})

print(f'Model artifacts: {estimator.model_data}')
'''

print('SageMaker training cell is commented out.')
print('Uncomment and update S3 paths to launch a cloud training job.')

---

## Summary

This notebook demonstrated the complete ML workflow:

1. ✅ **Data exploration** — Checked class distribution and image quality
2. ✅ **Augmentation** — Visualized what each transform does
3. ✅ **Model architecture** — Inspected layers, parameter counts, freeze status
4. ✅ **Training** — Local experiment with learning curve visualization
5. ✅ **Evaluation** — Confusion matrix and per-class analysis
6. ✅ **SageMaker** — Template for cloud-scale training

**Next steps:** Deploy the trained model using `make deploy` and test the API.